# Project 1 Module 7: Orchestration and Testing Practice

This workbook connects the individual ETL modules into one safe operating workflow. It focuses on stage order, command selection, test levels, evidence, and boundary-first troubleshooting.

## Safety contract

- Run from the `calgary-spatial-etl` repository with the `calgary-etl` kernel.
- Exercises are read-only unless a cell explicitly uses a temporary directory.
- The workbook does not run live Extract or write to PostGIS.
- Do not use skip flags to conceal a failed prerequisite.
- Attempt each exercise before reading or running its self-check.

## Learning objectives

By the end, you should be able to:

1. Put setup and ETL stages in a safe order.
2. Choose the correct `src.main` command for a scenario.
3. Distinguish unit, integration, smoke, and operational checks.
4. Interpret skipped database tests accurately.
5. Locate the earliest broken stage guarantee.
6. Assemble sufficient evidence before calling a run successful.

In [ ]:
from pathlib import Path
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in [candidate, *candidate.parents]:
        if (directory / "src/main.py").exists() and (directory / "environment.yml").exists():
            return directory
    raise FileNotFoundError("Run this notebook from inside calgary-spatial-etl.")


PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")
print(f"Interpreter: {sys.executable}")

## Part 1: Build the Safe Workflow

Put these stages in a safe operational order:

- automated tests
- Extract
- environment setup
- Git state inspection
- Load
- PostGIS setup
- post-load verification
- QA/QC
- review and commit
- Transform

Remember that setup and version control support ETL, while Extract, Transform, QA/QC, and Load process or govern data.

In [ ]:
# Replace TODO with the ordered stage names from the prompt.
workflow_order = ["TODO"]

required_stages = {
    "environment setup",
    "Git state inspection",
    "PostGIS setup",
    "Extract",
    "Transform",
    "QA/QC",
    "Load",
    "post-load verification",
    "automated tests",
    "review and commit",
}

assert set(workflow_order) == required_stages, "Use each stage exactly once."
for earlier, later in [
    ("environment setup", "Extract"),
    ("PostGIS setup", "Load"),
    ("Extract", "Transform"),
    ("Transform", "QA/QC"),
    ("QA/QC", "Load"),
    ("Load", "post-load verification"),
    ("automated tests", "review and commit"),
]:
    assert workflow_order.index(earlier) < workflow_order.index(later), f"{earlier} must precede {later}."
print("PASS: required dependencies are ordered safely.")

## Part 2: Choose the Orchestration Command

For each scenario, choose one command:

- `python -m src.main`
- `python -m src.main --skip-extract`
- `python -m src.main --skip-extract --skip-load`

Scenarios:

1. Capture fresh Calgary data and publish a complete verified database snapshot.
2. Reuse a deliberately preserved raw snapshot and republish all downstream outputs.
3. Reuse raw data, regenerate processed files, and stop after QA for inspection.
4. Extract failed, but old raw files remain. Decide whether any skip command should be used and explain why.

Write your reasoning in a Markdown cell before completing the dictionary below.

In [ ]:
command_choices = {
    1: "TODO",
    2: "TODO",
    3: "TODO",
    4: "TODO",
}

expected_commands = {
    1: "python -m src.main",
    2: "python -m src.main --skip-extract",
    3: "python -m src.main --skip-extract --skip-load",
    4: "do not bypass the failed prerequisite",
}
assert command_choices == expected_commands
print("PASS: command choices preserve the intended stage guarantees.")

## Part 3: Classify Verification Evidence

Classify each example as `unit`, `integration`, `smoke`, or `operational`:

1. `inspect_layer` rejects a temporary GeoJSON containing duplicate IDs.
2. A test loads a temporary layer into an isolated PostGIS schema and forces rollback.
3. The smallest representative workflow crosses all stage boundaries using controlled fixtures.
4. A fresh live Calgary run loads all current layers and direct SQL confirms counts, SRIDs, and indexes.

Then explain:

- Why a skipped PostGIS integration test is not a pass.
- Why live operational success does not replace deterministic regression tests.
- Why happy-path and failure-path checks are both necessary.

In [ ]:
verification_types = {
    1: "TODO",
    2: "TODO",
    3: "TODO",
    4: "TODO",
}

assert verification_types == {
    1: "unit",
    2: "integration",
    3: "smoke",
    4: "operational",
}
print("PASS: verification levels are classified correctly.")

## Part 4: Run Deterministic Tests Safely

The next cell runs only the QA unit-test module. It does not contact Calgary or PostGIS.

Before running it, predict:

1. How many tests should run?
2. Which failure behaviors are deliberately tested?
3. Which important behavior remains unverified because database integration is excluded?

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "unittest", "tests.test_qa", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(test_result.stderr or test_result.stdout)
assert test_result.returncode == 0, "The deterministic QA tests failed."
assert "Ran 3 tests" in (test_result.stderr + test_result.stdout)
print("PASS: three deterministic QA tests completed successfully.")

## Part 5: Diagnose the Earliest Broken Guarantee

For each symptom, name the first boundary to inspect and the first evidence to collect:

1. `ModuleNotFoundError: geopandas`
2. Extract receives HTTP 503.
3. Transform reports a large unexpected row decrease.
4. QA reports the wrong CRS.
5. Load cannot connect to `localhost:5433`.
6. Loaded and processed row counts differ.
7. Unit tests pass but integration tests are skipped.
8. `git status` shows a credential file.
9. A processed file exists even though the current Transform log records an error.

Use this sentence:

> The first broken guarantee is at the ___ boundary, shown by ___ evidence, so I will inspect ___ before changing downstream stages.

In [ ]:
# Fill each value with the earliest owning boundary.
boundary_answers = {
    1: "TODO",
    2: "TODO",
    3: "TODO",
    4: "TODO",
    5: "TODO",
    6: "TODO",
    7: "TODO",
    8: "TODO",
    9: "TODO",
}

assert boundary_answers == {
    1: "environment",
    2: "Extract",
    3: "Transform",
    4: "Transform-to-QA",
    5: "PostGIS setup",
    6: "Load",
    7: "integration verification",
    8: "Git safety",
    9: "Transform",
}
print("PASS: each symptom is assigned to its earliest owning boundary.")

## Part 6: Assemble Success Evidence

A console message alone is not sufficient evidence. For each stage, list the artifact or observation you would inspect:

| Stage | Evidence to identify |
|---|---|
| Environment | interpreter and import evidence |
| Git | working-tree and staged-diff evidence |
| PostGIS setup | service, database, and extension evidence |
| Extract | raw snapshot and provenance evidence |
| Transform | row, schema, geometry, and CRS evidence |
| QA/QC | blocking result and report evidence |
| Load | transaction, count, SRID, and index evidence |
| Tests | passed, failed, skipped, and errored evidence |

Then answer:

1. What evidence establishes that implementation is **complete**?
2. What additional evidence establishes that it is currently **operational**?
3. Which external prerequisites prevent either claim from being permanent?

In [ ]:
artifact_paths = {
    "extract_log": PROJECT_ROOT / "outputs/logs/extract_log.csv",
    "transform_log": PROJECT_ROOT / "outputs/logs/transform_log.csv",
    "qa_report": PROJECT_ROOT / "outputs/qa/qa_report.csv",
}
processed_paths = sorted((PROJECT_ROOT / "data/processed").glob("*.geojson"))

for name, path in artifact_paths.items():
    print(f"{name}: {'present' if path.exists() else 'missing'} -> {path.relative_to(PROJECT_ROOT)}")
print(f"processed GeoJSON files: {len(processed_paths)}")

# Presence is evidence to inspect, not proof that an artifact belongs to the latest run.
assert set(artifact_paths) == {"extract_log", "transform_log", "qa_report"}
print("PASS: expected operational artifacts have been located without modifying them.")

## Capstone: Write an Operator Runbook

Without copying commands from the study guide, write a concise runbook for a fresh verified pipeline run. Include:

1. Environment and interpreter checks.
2. Git state inspection.
3. PostGIS startup and initialization.
4. The full pipeline command.
5. Stage logs and artifacts to inspect.
6. Automated unit and integration test commands.
7. Direct database evidence required after Load.
8. The stop condition at every failed stage.
9. The final Git review and commit process.
10. The distinction between a complete implementation and a currently operational run.

## Mastery check

You are ready to move to the cumulative assessment when you can:

- choose commands from scenarios rather than memorizing one command
- identify the earliest failed guarantee
- explain what each test level proves and does not prove
- distinguish skipped tests from passing tests
- assemble evidence across files, logs, QA, PostGIS, and Git
- explain why stale artifacts must not be mistaken for current success

## AI Practice Review

After completing the working copy, save it and ask Copilot:

> Review my completed practice notebook without assigning a progression grade. Preserve my original answers and code. Inspect my reasoning, implementations, self-checks, errors, and outputs. Cite evidence for demonstrated strengths and misconceptions, recommend the smallest useful exercises to retry, and ask targeted follow-up questions before giving complete corrected answers.

After reviewing the feedback, preserve the completed notebook with `python scripts/save_attempt.py 7`.